# ARISE — 02 · Attribute Anomaly Detection (Graph Contrastive Learning)

This is the **attribute anomaly module** (paper Section IV-B2). It follows the
CoLA node-vs-subgraph contrastive scheme:

- For a target node $v_i$ we sample a local **subgraph** by random walk.
- **Positive pair** = $(v_i,\text{ its own subgraph})$; **negative pair** = $(v_i,\text{ another node's subgraph})$.
- The target's row inside the subgraph is **masked to 0** so the discriminator can't cheat.
- A 1-layer **GCN** encodes the subgraph → mean **Readout** $e_i$ (Eq. 4); the target embedding is $z_i=\mathrm{ReLU}(x_iW)$ (Eq. 5, shared $W$); a **Bilinear** discriminator scores the pair (Eq. 6); **BCE** loss trains it (Eq. 7).
- Attribute anomaly score: $a_i = s_{i,neg}-s_{i,pos}$ averaged over rounds (Eq. 10-11).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # make the `arise` package importable
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
from arise.data import load_dataset, inject_anomalies, PAPER_INJECTION
from arise.pipeline import normalize_features

graph = load_dataset("cora")
graph = inject_anomalies(graph, seed=42, **PAPER_INJECTION["Cora"])
# L1 row-normalise bag-of-words features (standard GCN/CoLA preprocessing)
gm = normalize_features(graph)
print(graph.summary())

## 1. Subgraph sampling (random walk)
Each node gets a fixed-size subgraph (size 4 in the paper). The anchor is index 0.

In [ ]:
from arise.model import build_subgraphs

subgraphs = build_subgraphs(graph.adj, subgraph_size=4, seed=0)
print("subgraph of node 0:", subgraphs[0])
print("shape:", subgraphs.shape)

## 2. The contrastive model
1-layer GCN encoder (shared weight $W$) + Bilinear discriminator.

In [ ]:
from arise.model import ARISEContrastive
model = ARISEContrastive(in_dim=graph.num_features, hidden_dim=64)
print(model)

## 3. Train (Eq. 7 BCE on positive/negative pairs)

In [ ]:
from arise.pipeline import train_contrastive

model, history = train_contrastive(gm, hidden_dim=64, epochs=60, lr=0.003,
                                   weight_decay=1e-5, batch_size=300, seed=0, verbose=True)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history, color="#6ea8ff")
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.title("Contrastive training loss"); plt.show()

## 4. Attribute anomaly scores (Eq. 10-11)
Multi-round: resample subgraphs each round, score $a_i = s_{neg}-s_{pos}$, average.

In [ ]:
from arise.pipeline import attribute_anomaly_scores
from arise.metrics import evaluate

score_a = attribute_anomaly_scores(gm, model, rounds=4, seed=0)

# How well does it find the *attribute* anomalies specifically?
print("attribute module on attribute anomalies:",
      evaluate(graph.attr_mask.astype(int), score_a))

### Score distribution: anomalies should score higher

In [ ]:
plt.figure(figsize=(7,4))
m = graph.attr_mask
plt.hist(score_a[~m], bins=40, alpha=.6, label="normal", color="#6ea8ff", density=True)
plt.hist(score_a[m], bins=20, alpha=.8, label="attribute anomaly", color="#ffb454", density=True)
plt.xlabel("attribute anomaly score"); plt.ylabel("density"); plt.legend()
plt.title("Attribute score separation"); plt.show()

## Summary
- Trained the GCN contrastive network; loss decreases steadily.
- The attribute module ranks injected **attribute anomalies** well above normal nodes.
- Crucially, the **node embeddings** $z_i$ learned here feed the topology module next.

➡️ Next: **03** uses these embeddings to detect *topology* anomalies via substructure similarity.